# 01 · Exploración de datos (sección 5.3 del informe)

Este notebook reproduce las cifras, tablas y figuras de la sección **5.3 Exploración de datos** (fase *Comprensión de los datos* de CRISP-DM, objetivo específico 2). Es un análisis descriptivo: no ajusta modelos y no modifica el archivo de origen. La limpieza formal y la carga a la base de datos se hacen en 5.4.

**Requisitos:** `pandas`, `numpy`, `matplotlib` y `openpyxl`. El archivo `HISTORICOS_DE_VENTAS.xlsx` debe estar en `data/raw/` (no se versiona por confidencialidad).

**Alcance:** solo los productos con historial completo, es decir, los que aparecen en las cinco hojas (VTAS 2022 a VTAS 2026), y los 57 meses de enero de 2022 a septiembre de 2026. Los demás productos vigentes no se analizan. Las series agregadas se expresan en piezas (productos con `unidad = PZA`).

Las tablas y figuras llevan la numeración del informe (Tablas 4 a 9, Figuras 4 a 6). Las figuras se guardan en `notebooks/figures/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# --- Parámetros ---------------------------------------------------------
RUTA_EXCEL = Path("../data/raw/HISTORICOS_DE_VENTAS.xlsx")
CARPETA_FIGURAS = Path("figures")
CARPETA_FIGURAS.mkdir(exist_ok=True)

MESES = ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]
MESES_ES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
ANIOS = [2022, 2023, 2024, 2025, 2026]
ULTIMO_MES = "2026-09-01"                    # corte: octubre a diciembre es el horizonte a estimar
TEMPORADA_ALTA = {12, 1, 2, 3, 4, 7, 8}      # definición preliminar de la empresa (5.1.3)
PEDIDO_CONFIRMADO = ("8419", "2022-12-01")   # kit de temazcal, pedido grande confirmado

AZUL, GRIS = "#2b78d6", "#bdbdb0"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})


def mostrar(tabla, formato=None):
    """Muestra una tabla sin el índice, lista para capturar."""
    display(tabla.style.hide(axis="index").format(formato or {}, na_rep="–"))


def etiqueta_mes(fecha):
    return f"{MESES_ES[fecha.month - 1]}-{fecha.year}"

## 5.3.1 Alcance, carga y estructuración mínima

Se lee cada hoja (`VTAS 2022` a `VTAS 2026`): la fila 1 es `AÑO: 20XX`, la fila 2 es el encabezado y la última fila trae el total declarado (`TOTAL: N`). La fila de totales se separa para usarla en los controles de 5.3.2.

In [ ]:
hojas, totales = {}, {}
for anio in ANIOS:
    df = pd.read_excel(RUTA_EXCEL, sheet_name=f"VTAS {anio}", header=1, dtype={"CLAVE": object})
    es_total = df["CLAVE"].astype(str).str.startswith("TOTAL")
    totales[anio] = df[es_total].iloc[0]
    df = df[~es_total].copy()
    df["clave_numerica"] = df["CLAVE"].map(lambda x: isinstance(x, (int, float)))   # tipo original
    df["clave"] = df["CLAVE"].astype(str).str.strip().str.upper()                   # clave como texto
    df[MESES + ["TOTAL"]] = df[MESES + ["TOTAL"]].apply(pd.to_numeric, errors="coerce")
    hojas[anio] = df

print({anio: len(h) for anio, h in hojas.items()})

**Vigentes, historial completo y unidad.** Se cargan todos los productos de la hoja `VTAS 2026` (los vigentes). Un producto tiene historial completo si su clave aparece en las cinco hojas, de 2022 a 2026; se guardan todos y, al analizar, se filtra por `historial_completo = 1`. La unidad es KG si la clave tiene algún valor decimal en cualquier hoja.

In [ ]:
vigentes = hojas[2026]["clave"].tolist()

# historial_completo: la clave aparece en las cinco hojas (2022 a 2026)
en_todas = set.intersection(*[set(h["clave"]) for h in hojas.values()])
# para describir a los excluidos: ¿aparecen en alguna hoja de 2022 a 2025?
claves_previas = set().union(*[set(hojas[a]["clave"]) for a in range(2022, 2026)])

# unidad: KG si la clave tiene algún valor con decimales en cualquier hoja
con_decimales = set()
for h in hojas.values():
    v = h[h["clave"].isin(vigentes)]
    con_decimales |= set(v.loc[(v[MESES].round(2) % 1 != 0).any(axis=1), "clave"])

productos = pd.DataFrame({
    "clave": vigentes,
    "descripcion": hojas[2026]["DESCRIPCION"].str.strip().tolist(),
    "historial_completo": [int(c in en_todas) for c in vigentes],
    "unidad": ["KG" if c in con_decimales else "PZA" for c in vigentes],
})

# Universo de análisis: productos con historial completo
universo = productos[productos["historial_completo"] == 1].reset_index(drop=True)
claves = universo["clave"].tolist()
productos.head()

In [ ]:
# Formato largo: una fila por producto y mes, con fecha ISO (siempre día 01)
largo = pd.concat([
    h.melt(id_vars="clave", value_vars=MESES, var_name="mes", value_name="cantidad")
     .assign(anio=anio)
    for anio, h in hojas.items()
])
largo["fecha"] = pd.to_datetime({"year": largo["anio"],
                                 "month": largo["mes"].map({m: i + 1 for i, m in enumerate(MESES)}),
                                 "day": 1})

# Solo vigentes y hasta el mes de corte
largo = largo[largo["clave"].isin(vigentes) & (largo["fecha"] <= ULTIMO_MES)]
fechas = pd.date_range("2022-01-01", ULTIMO_MES, freq="MS")

# Matriz producto x mes de los vigentes (para describir el alcance en la Tabla 5)
matriz_vig = (largo.pivot_table(index="clave", columns="fecha", values="cantidad", aggfunc="sum")
                   .reindex(index=vigentes, columns=fechas)
                   .fillna(0))

matriz_vig.index.name, matriz_vig.columns.name = "clave", "fecha"

# Tabla ventas (se guardan todos los vigentes): clave, fecha, cantidad (llave: clave + fecha)
ventas = matriz_vig.stack().rename("cantidad").reset_index()
ventas["fecha"] = ventas["fecha"].dt.strftime("%Y-%m-%d")
print(f"ventas: {len(ventas):,} filas ({len(vigentes)} productos x {len(fechas)} meses)")

# Matriz de análisis: se filtra por historial_completo = 1 (productos en las cinco hojas)
matriz = matriz_vig.loc[claves]
ventas.head()

**Tabla 4.** Variables derivadas de cada producto.

In [ ]:
tabla4 = pd.DataFrame({
    "Variable": ["historial_completo", "unidad"],
    "Valores": ["1 / 0", "PZA / KG"],
    "Regla": ["1 si la clave aparece en las cinco hojas (VTAS 2022 a VTAS 2026)",
              "KG si la clave tiene algún valor decimal (a dos decimales)"],
    "Resultado": [f"{(productos.historial_completo == 1).sum()} con valor 1 (se analizan) y {(productos.historial_completo == 0).sum()} con valor 0 (se guardan, no se analizan)",
                  f"En los {len(universo)} analizados: {(universo.unidad == 'PZA').sum()} PZA y {(universo.unidad == 'KG').sum()} KG"],
})
mostrar(tabla4)

**Tabla 5.** Productos vigentes según su historial y piezas que representan (enero de 2022 a septiembre de 2026).

In [ ]:
# Piezas por producto (solo unidad PZA) para dimensionar lo que queda fuera
piezas_prod = matriz_vig.sum(axis=1).to_numpy() * (productos["unidad"] == "PZA").to_numpy()
completo = productos["historial_completo"].to_numpy() == 1
nuevo = ~completo & ~productos["clave"].isin(claves_previas).to_numpy()   # solo aparecen en 2026
parcial = ~completo & ~nuevo                                               # en 2026 y en algunas, no todas, de 2022 a 2025

grupos = [("Vigentes (hoja VTAS 2026)", np.ones(len(productos), bool)),
          ("Con historial completo: se analizan", completo),
          ("Sin historial completo: solo aparecen en 2026 (nuevos)", nuevo),
          ("Sin historial completo: faltan en alguna hoja de 2022 a 2025", parcial)]
tabla5 = pd.DataFrame({
    "Grupo": [g for g, _ in grupos],
    "Productos": [int(m.sum()) for _, m in grupos],
    "Piezas": [piezas_prod[m].sum() for _, m in grupos],
})
tabla5["% de las piezas"] = 100 * tabla5["Piezas"] / piezas_prod.sum()
mostrar(tabla5, {"Piezas": "{:,.0f}", "% de las piezas": "{:.1f}%"})

In [ ]:
G = matriz.to_numpy(dtype=float)              # productos x meses (57)
n_prod, T = G.shape
es_pza = (universo["unidad"] == "PZA").to_numpy()
es_kg = ~es_pza

total_obs = G.size
con_venta = int((G > 0).sum())
print(f"Observaciones producto-mes: {total_obs:,} ({n_prod} x {T})")
print(f"Meses analizados: {T} ({fechas[0]:%Y-%m} a {fechas[-1]:%Y-%m})")
print(f"  con venta: {con_venta:,}   en cero: {total_obs - con_venta:,}")

kg_2025 = G[es_kg][:, fechas.year == 2025].sum()
print(f"Productos en kilogramos: {es_kg.sum()}  ->  {kg_2025:,.0f} kg en 2025")
print("¿Todos los KG son a granel?", universo.loc[es_kg, "descripcion"].str.contains("GRANEL", case=False).all())

# Octubre de 2026 (incompleto) frente a los octubres anteriores, en piezas, desde las hojas originales
oct_prev = [hojas[a].loc[hojas[a]["clave"].isin(universo.loc[es_pza, "clave"]), "OCT"].sum() for a in range(2022, 2026)]
oct_26 = hojas[2026].loc[hojas[2026]["clave"].isin(universo.loc[es_pza, "clave"]), "OCT"].sum()
print(f"Octubre 2026 (parcial): {oct_26:,.0f} piezas vs promedio de octubres 2022-2025: {np.mean(oct_prev):,.0f}")

> **Nota.** Las diferencias de abril y mayo de 2026 frente a la exportación anterior (−1,504 y −256 unidades) se obtuvieron comparando dos archivos. Aquí solo existe el archivo definitivo, así que esa comparación no se puede recalcular; se toma como dato definitivo la última exportación.

## 5.3.2 Calidad e identidad de los datos

Controles por hoja: el número de filas debe coincidir con el total declarado; la suma de cada mes, con la fila de totales; y la suma de los 12 meses de cada producto, con su columna `TOTAL`.

**Tabla 6.** Controles de calidad por hoja.

In [ ]:
filas = []
for anio, h in hojas.items():
    tot = totales[anio]
    filas.append({
        "Hoja": f"VTAS {anio}",
        "Filas": len(h),
        "Total declarado": int(str(tot["CLAVE"]).split(":")[1]),
        "Dif. máx. por mes": (h[MESES].sum() - tot[MESES].astype(float)).abs().max(),
        "Dif. máx. por fila": (h[MESES].sum(axis=1) - h["TOTAL"]).abs().max(),
        "Claves duplicadas": int(h["clave"].duplicated().sum()),
        "Celdas vacías o negativas": int(h[MESES].isna().sum().sum() + (h[MESES] < 0).sum().sum()),
    })
mostrar(pd.DataFrame(filas), {"Dif. máx. por mes": "{:.2f}", "Dif. máx. por fila": "{:.2f}"})

In [ ]:
# Identidad de los productos
h26 = hojas[2026][hojas[2026]["clave"].isin(claves)]       # solo productos analizados
print(f"Claves 2026: {int(h26['clave_numerica'].sum())} numéricas y {int((~h26['clave_numerica']).sum())} alfanuméricas")
print("Ejemplos de claves alfanuméricas:", h26.loc[~h26["clave_numerica"], "clave"].head(3).tolist())

# Descripciones distintas por clave entre años (solo vigentes)
desc = pd.concat([h.loc[h["clave"].isin(claves), ["clave", "DESCRIPCION"]] for h in hojas.values()])
desc["d"] = desc["DESCRIPCION"].str.strip().str.upper()
cambian = desc.groupby("clave")["d"].nunique() > 1
print("Productos analizados con más de una descripción entre años:", int(cambian.sum()))
ej = desc[desc["clave"].isin(cambian[cambian].index)].drop_duplicates(["clave", "d"]).groupby("clave")["DESCRIPCION"].agg(["first", "last"]).head(3)
mostrar(ej.reset_index().rename(columns={"first": "descripción más antigua", "last": "descripción de 2026"}))

# Descripciones repetidas en claves distintas dentro de 2026
d26 = h26.assign(d=h26["DESCRIPCION"].str.strip().str.upper())
repetidas = d26[d26["d"].duplicated(keep=False)].sort_values("d")
repetidas = repetidas.assign(unidades=repetidas["clave"].map(matriz.sum(axis=1)))
if repetidas.empty:
    print("Descripciones repetidas en claves distintas dentro de 2026 (analizados): ninguna")
else:
    mostrar(repetidas[["clave", "DESCRIPCION", "unidades"]], {"unidades": "{:,.0f}"})

## 5.3.3 Ceros e historial

Un cero es una venta real de cero unidades: no se imputa ni se elimina. La "primera venta" de cada producto es el primer mes con cantidad mayor a cero.

In [ ]:
hay_venta = G > 0
primera = np.where(hay_venta.any(axis=1), hay_venta.argmax(axis=1), -1)     # -1 = sin ventas en el periodo
tiene = primera >= 0
meses_hist = T - primera                                                     # meses desde la primera venta

# Ceros sobre todas las observaciones y desde la primera venta
ventana = int(meses_hist[tiene].sum())
ceros_ventana = ventana - int(hay_venta.sum())
print(f"Ceros sobre todas las observaciones: {total_obs - con_venta:,} de {total_obs:,} ({100 * (total_obs - con_venta) / total_obs:.1f}%)")
print(f"Ceros desde la primera venta:        {ceros_ventana:,} de {ventana:,} ({100 * ceros_ventana / ventana:.1f}%)")

# Proporción de ceros por producto (>= 12 meses de historia)
largo_ok = tiene & (meses_hist >= 12)
prop_ceros = np.array([1 - hay_venta[i, primera[i]:].mean() for i in np.where(largo_ok)[0]])
q1, mediana, q3 = np.percentile(prop_ceros, [25, 50, 75])
print(f"\nProductos con >= 12 meses de historia: {largo_ok.sum()}")
print(f"  proporción de ceros: mediana {mediana:.0%} (cuartiles {q1:.0%} y {q3:.0%}); con más de 50%: {(prop_ceros > 0.5).sum()}")

In [ ]:
def racha_ceros(v):
    mejor = actual = 0
    for x in v:
        actual = 0 if x else actual + 1
        mejor = max(mejor, actual)
    return mejor

print("Primera venta en ene-2022 (inicio real desconocido):", int((primera == 0).sum()))
print("Productos sin ventas dentro del corte (solo vendieron en octubre de 2026):", int((~tiene).sum()))
print("Con una racha de >= 6 meses sin venta tras su primera venta:",
      sum(racha_ceros(hay_venta[i, primera[i]:]) >= 6 for i in np.where(tiene)[0]))
print("Sin ventas de abril a septiembre de 2026:", int((G[:, fechas.get_loc(pd.Timestamp('2026-04-01')):].sum(axis=1) == 0).sum()))

**Tabla 7.** Proporción de meses con venta cero por producto, desde su primera venta hasta septiembre de 2026.

In [ ]:
prop_todos = np.array([1 - hay_venta[i, primera[i]:].mean() for i in np.where(tiene)[0]])
rangos = pd.cut(prop_todos, bins=[-0.001, 0, 0.25, 0.50, 1.0],
                labels=["0% (vende todos los meses)", "más de 0% y hasta 25%", "más de 25% y hasta 50%", "más de 50%"])
tabla7 = rangos.value_counts().sort_index().rename_axis("Meses con venta cero").reset_index(name="Productos")
mostrar(tabla7)

> **Productos nuevos y con historia parcial.** Los productos sin historial completo (Tabla 5) no se analizan ni se modelan: su historia es corta o tiene huecos, y completarla exigiría imputar.

## 5.3.4 Valores atípicos

Regla del rango intercuartílico (Tukey, 1977): en cada producto, un mes es atípico si supera **Q3 + 3 × RIC**, calculados solo sobre los meses con venta (los ceros se excluyen). Se evalúan los productos con al menos 8 meses con venta.

Clasificación: **sostenido** si el mes contiguo también es atípico; **recurrente** si el mismo producto es atípico en el mismo mes de otro año; **aislado** en los demás casos.

In [ ]:
atipico = np.zeros_like(G, dtype=bool)
umbral, mediana_prod = {}, {}
for i in range(n_prod):
    if not tiene[i]:
        continue
    s = G[i, primera[i]:]
    s = s[s > 0]
    if len(s) < 8:
        continue
    q1, q3 = np.percentile(s, [25, 75])
    umbral[i] = q3 + 3 * (q3 - q1)
    mediana_prod[i] = np.median(s)
    atipico[i] = G[i] > umbral[i]

puntos = np.argwhere(atipico)                       # (producto, mes)
evaluables = list(umbral)
S = G[es_pza].sum(axis=0)                           # piezas por mes (serie cruda)

print(f"Productos evaluables: {len(evaluables)}")
print(f"Atípicos: {len(puntos)} en {int(atipico.any(axis=1).sum())} productos "
      f"({100 * len(puntos) / hay_venta[evaluables].sum():.2f}% de las observaciones con venta)")
print(f"En piezas: {int((atipico & es_pza[:, None]).sum())} puntos = {100 * G[atipico & es_pza[:, None]].sum() / S.sum():.1f}% de las unidades vendidas")

# Clasificación
clases = []
for i, t in puntos:
    if (t > 0 and atipico[i, t - 1]) or (t < T - 1 and atipico[i, t + 1]):
        c = "Sostenido"
    elif any(atipico[i, u] for u in range(t % 12, T, 12) if u != t):
        c = "Recurrente"
    else:
        c = "Aislado"
    clases.append(c)
atip = pd.DataFrame(puntos, columns=["i", "t"]).assign(clase=clases)
atip["anio"] = fechas[atip["t"]].year

print("\nPor clase:", atip["clase"].value_counts().to_dict())
print("Por año:  ", atip["anio"].value_counts().sort_index().to_dict())

**Tabla 8.** Principales valores atípicos en piezas, ordenados por su exceso sobre la mediana del producto.

In [ ]:
filas = []
for _, r in atip.iterrows():
    i, t = int(r.i), int(r.t)
    if not es_pza[i]:
        continue
    filas.append({
        "Clave": claves[i],
        "Producto": universo.loc[i, "descripcion"],
        "Mes": etiqueta_mes(fechas[t]),
        "Piezas": G[i, t],
        "Mediana mensual": mediana_prod[i],
        "% de las piezas del mes": 100 * G[i, t] / S[t],
        "Clase": r.clase,
        "exceso": G[i, t] - mediana_prod[i],
    })
tabla8 = pd.DataFrame(filas).sort_values("exceso", ascending=False).head(8).drop(columns="exceso")
es_kit = (tabla8["Clave"] == PEDIDO_CONFIRMADO[0]) & (tabla8["Mes"] == "dic-2022")
tabla8.loc[es_kit, "Clase"] += "*"
mostrar(tabla8, {"Piezas": "{:,.0f}", "Mediana mensual": "{:,.1f}", "% de las piezas del mes": "{:.1f}%"})
print("* Pedido grande confirmado por la empresa.")

**Serie ajustada.** Para medir el efecto de los atípicos se sustituye por la mediana del producto cada atípico *aislado* y el pedido confirmado del kit. Los sostenidos y recurrentes no se sustituyen, porque pueden ser cambios de nivel o estacionalidad. La cantidad original no se modifica: el ajuste vive en una copia (`A`).

In [ ]:
i_kit = claves.index(PEDIDO_CONFIRMADO[0])
t_kit = fechas.get_loc(pd.Timestamp(PEDIDO_CONFIRMADO[1]))

A = G.copy()
reemplazos = 0
for _, r in atip.iterrows():
    i, t = int(r.i), int(r.t)
    if r.clase == "Aislado" or (i == i_kit and t == t_kit):
        A[i, t] = mediana_prod[i]
        reemplazos += 1
SA = A[es_pza].sum(axis=0)                          # piezas por mes (serie ajustada)

reemp_pza = int(((A != G) & es_pza[:, None]).sum())
retirado = 100 * (S.sum() - SA.sum()) / S.sum()
print(f"Celdas sustituidas: {reemplazos} (en piezas: {reemp_pza}); unidades retiradas: {retirado:.1f}% de las piezas")

In [ ]:
# Efecto en el promedio del mismo mes (modelo base), con los años completos 2022-2025
efecto = pd.DataFrame({
    "Mes": MESES_ES,
    "Promedio crudo": [S[m:48:12].mean() for m in range(12)],
    "Promedio ajustado": [SA[m:48:12].mean() for m in range(12)],
})
efecto["Variación"] = 100 * (efecto["Promedio ajustado"] / efecto["Promedio crudo"] - 1)
mostrar(efecto, {"Promedio crudo": "{:,.0f}", "Promedio ajustado": "{:,.0f}", "Variación": "{:.1f}%"})

# A nivel de producto: promedio del mismo mes con y sin el atípico (atípicos de 2022-2025, en piezas)
infl = []
for _, r in atip.iterrows():
    i, t = int(r.i), int(r.t)
    if t >= 48 or not es_pza[i]:
        continue
    m = t % 12
    anios = [y for y in range(4) if 2022 + y >= fechas[primera[i]].year]    # desde su primera venta
    crudo = np.mean([G[i, y * 12 + m] for y in anios])
    ajust = np.mean([A[i, y * 12 + m] for y in anios])
    if ajust > 0:
        infl.append(crudo / ajust - 1)
infl = np.array(infl)
print(f"\nCasos producto-mes: {len(infl)}; con el promedio crudo >50% por encima del ajustado: "
      f"{(infl > 0.5).sum()} ({100 * (infl > 0.5).mean():.0f}%); mediana de inflación: {100 * np.median(infl):.0f}%")

dic_kit = [G[i_kit, y * 12 + 11] for y in range(4)]
dic_kit_aj = [A[i_kit, y * 12 + 11] for y in range(4)]
print(f"Kit 8419, promedio de diciembre: {np.mean(dic_kit):,.0f} crudo vs {np.mean(dic_kit_aj):,.0f} ajustado")

## 5.3.5 Tendencia y estacionalidad

Todas las cifras de esta sección están en piezas (productos con `unidad = PZA`).

### Tendencia

In [ ]:
filas = []
for y in ANIOS:
    en_anio = fechas.year == y
    ene_sep = np.isin(fechas.month, range(1, 10)) & en_anio
    pza_anio = G[es_pza][:, en_anio]
    filas.append({
        "Año": y,
        "Piezas del año": S[en_anio].sum(),
        "Piezas ene-sep": S[ene_sep].sum(),
        "Productos con venta": int((G[:, en_anio].sum(axis=1) > 0).sum()),
        "Piezas por producto activo": S[en_anio].sum() / (pza_anio.sum(axis=1) > 0).sum(),
    })
tend = pd.DataFrame(filas)
tend["Var. ene-sep"] = tend["Piezas ene-sep"].pct_change() * 100
tend.loc[tend["Año"] == 2026, ["Piezas del año", "Piezas por producto activo"]] = np.nan   # 2026 está incompleto
mostrar(tend, {"Piezas del año": "{:,.0f}", "Piezas ene-sep": "{:,.0f}", "Piezas por producto activo": "{:,.0f}",
               "Var. ene-sep": "{:+.1f}%"})

In [ ]:
# Contexto: vigentes sin historial completo (no se analizan), en piezas, enero a septiembre
fuera_pza = ~completo & (productos["unidad"] == "PZA").to_numpy()
for y in (2025, 2026):
    cols = [(f, i) for i, f in enumerate(fechas) if f.year == y and f.month <= 9]
    fuera_y = matriz_vig.to_numpy()[fuera_pza][:, [i for _, i in cols]].sum()
    dentro_y = S[[i for _, i in cols]].sum()
    print(f"Ene-sep {y}: sin historial completo {fuera_y:,.0f} piezas; los analizados son el {100 * dentro_y / (dentro_y + fuera_y):.1f}% del total")

# Suma móvil de 12 meses a septiembre
suma12 = {y: S[(y - 2022) * 12 - 3:(y - 2022) * 12 + 9].sum() for y in range(2023, 2027)}
print("Suma de 12 meses a septiembre:", {y: f"{v:,.0f}" for y, v in suma12.items()})
print(f"  variación del último año: {100 * (suma12[2026] / suma12[2025] - 1):+.1f}%")
print(f"Cuarto trimestre de 2025: {S[45:48].sum():,.0f} piezas")
print("Diciembre crudo por año:", [f"{S[y * 12 + 11]:,.0f}" for y in range(4)])

**Figura 4.** Ventas mensuales de los productos vigentes en piezas y su media móvil de 12 meses.

In [ ]:
ma12 = pd.Series(S).rolling(12).mean().to_numpy()
x = np.arange(T)

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(x, S, color=GRIS, lw=1.6, label="Ventas mensuales")
ax.plot(x, ma12, color=AZUL, lw=2.6, label="Media móvil de 12 meses")

# Pedido del kit (dic-2022) y máximo de dic-2025
ax.annotate("Pedido del kit\n(dic-2022)", xy=(t_kit, S[t_kit]), xytext=(t_kit + 4, S[t_kit] - 400),
            fontsize=9, arrowprops=dict(arrowstyle="-", color="#777"))

# Etiquetas al final de cada serie (margen derecho)
ax.annotate(f"{S[-1]:,.0f}", xy=(T - 1, S[-1]), xytext=(8, -4), textcoords="offset points", fontsize=9, color="#666")
ax.annotate(f"{ma12[-1]:,.0f}", xy=(T - 1, ma12[-1]), xytext=(8, 4), textcoords="offset points", fontsize=9, color=AZUL)

ax.set_xticks(range(0, T, 6))
ax.set_xticklabels([etiqueta_mes(fechas[t]) for t in range(0, T, 6)])
ax.set_ylabel("Piezas")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.set_xlim(-1, T - 1)
ax.set_title("Ventas mensuales de los productos con historial completo", loc="left", fontweight="bold")
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
fig.savefig(CARPETA_FIGURAS / "figura_4_ventas_mensuales.png", dpi=200)
plt.show()

### Estacionalidad

Para cada mes se calcula el cociente entre su promedio (años completos 2022 a 2025) y el promedio mensual de esos cuatro años, sobre la serie ajustada.

In [ ]:
promedio_total = SA[:48].mean()
indice = np.array([SA[m:48:12].mean() / promedio_total for m in range(12)])
indice_crudo = np.array([S[m:48:12].mean() / S[:48].mean() for m in range(12)])
anios_sobre = [sum(SA[y * 12 + m] / SA[y * 12:y * 12 + 12].mean() > 1 for y in range(4)) for m in range(12)]

tabla_ind = pd.DataFrame({
    "Mes": MESES_ES,
    "Temporada (empresa)": ["alta" if m in TEMPORADA_ALTA else "baja" for m in range(1, 13)],
    "Índice ajustado": indice,
    "Índice sin ajustar": indice_crudo,
    "Años (de 4) sobre el promedio": anios_sobre,
})
mostrar(tabla_ind, {"Índice ajustado": "{:.2f}", "Índice sin ajustar": "{:.2f}"})

In [ ]:
# Temporada alta vs. baja (definición preliminar de la empresa)
alta = [m - 1 for m in TEMPORADA_ALTA]
def prom(y, meses):
    return np.mean([SA[y * 12 + m] for m in meses])
baja = [m for m in range(12) if m not in alta]

m_alta = np.mean([prom(y, alta) for y in range(4)])
m_baja = np.mean([prom(y, baja) for y in range(4)])
print(f"Promedio mensual en temporada alta: {m_alta:,.0f}  |  baja: {m_baja:,.0f}  |  razón: {m_alta / m_baja:.2f}")
print("Razón alta/baja por año (2022-2025):", [round(float(prom(y, alta) / prom(y, baja)), 2) for y in range(4)])

# Semana Santa: participación de marzo en la suma de marzo y abril
part_mar = [100 * SA[y * 12 + 2] / (SA[y * 12 + 2] + SA[y * 12 + 3]) for y in range(5)]
print("Participación de marzo en mar+abr (2022-2026):", [f"{v:.1f}%" for v in part_mar])

**Figura 5.** Índice estacional ajustado por mes (2022 a 2025) y clasificación de temporada propuesta por la empresa.

In [ ]:
es_alta = np.array([m in TEMPORADA_ALTA for m in range(1, 13)])
n_alta_sobre = int((indice[es_alta] > 1).sum())

fig, ax = plt.subplots(figsize=(10, 4.8))
colores = [AZUL if a else GRIS for a in es_alta]
barras = ax.bar(MESES_ES, indice - 1, color=colores, width=0.7)
ax.axhline(0, color="#444", lw=1)
for b, v in zip(barras, indice):
    ax.text(b.get_x() + b.get_width() / 2, (v - 1) + (0.012 if v >= 1 else -0.012), f"{v:.2f}",
            ha="center", va="bottom" if v >= 1 else "top", fontsize=9)

ticks = np.array([0.8, 0.9, 1.0, 1.1, 1.2])
ax.set_yticks(ticks - 1)
ax.set_yticklabels([f"{v:.1f}" for v in ticks])
ax.set_ylim(-0.30, 0.30)
ax.set_ylabel("Índice (1.0 = promedio mensual)")
ax.set_title(f"{n_alta_sobre} de los {es_alta.sum()} meses de temporada alta superan el promedio mensual",
             loc="left", fontweight="bold")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=AZUL, label="Temporada alta (empresa)"), Patch(color=GRIS, label="Temporada baja (empresa)")],
          frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig(CARPETA_FIGURAS / "figura_5_indice_estacional.png", dpi=200)
plt.show()

## 5.3.6 Concentración de las ventas

Se ordenan los productos en piezas por las unidades vendidas en los últimos 12 meses cerrados (octubre de 2025 a septiembre de 2026) y se acumula su participación.

In [ ]:
ultimos12 = G[es_pza][:, T - 12:].sum(axis=1)
p = np.sort(ultimos12[ultimos12 > 0])[::-1]
acum = np.cumsum(p) / p.sum()
n_80 = int((acum < 0.80).sum() + 1)
n_95 = int((acum < 0.95).sum() + 1)

print(f"Productos en piezas con venta en los últimos 12 meses: {len(p)}")
print(f"  80% de las piezas: {n_80} productos ({100 * n_80 / len(p):.0f}%)   |   95%: {n_95} productos")
print(f"  top 10: {100 * acum[9]:.1f}%   top 20: {100 * acum[19]:.1f}%")
print(f"  mediana por producto: {np.median(p):.0f} piezas   |   con menos de 50: {(p < 50).sum()} ({100 * (p < 50).mean():.0f}%)")

**Figura 6.** Participación acumulada de las piezas vendidas por producto, de octubre de 2025 a septiembre de 2026.

In [ ]:
pct_prod = 100 * np.arange(1, len(p) + 1) / len(p)

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot([0, 100], [0, 100], color=GRIS, lw=1.4, ls="--", label="Reparto uniforme")
ax.plot(np.r_[0, pct_prod], np.r_[0, 100 * acum], color=AZUL, lw=2.6, label="Participación acumulada")
ax.axhline(80, color="#777", lw=0.9, ls=":")
ax.plot([100 * n_80 / len(p)], [80], "o", color=AZUL)
ax.annotate(f"{n_80} productos ({100 * n_80 / len(p):.0f}%)\naportan el 80%", xy=(100 * n_80 / len(p), 80),
            xytext=(38, 62), fontsize=9, arrowprops=dict(arrowstyle="-", color="#777"))
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.set_xlabel("% de los productos (de mayor a menor venta)")
ax.set_ylabel("% acumulado de las piezas")
ax.set_title(f"El {100 * n_80 / len(p):.0f}% de los productos concentra el 80% de las piezas", loc="left", fontweight="bold")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(CARPETA_FIGURAS / "figura_6_concentracion.png", dpi=200)
plt.show()

## 5.3.7 Registro de decisiones

**Tabla 9.** Hallazgos de la exploración y decisiones para las fases siguientes.

| Hallazgo | Decisión o propuesta | Se aplica en |
|---|---|---|
| Las cinco hojas cuadran; sin nulos, negativos ni duplicados | Se conservan los valores originales; los controles de 5.3.2 pasan a pruebas automáticas | 5.4 |
| La clave mezcla números y texto | Clave como texto; llave de ventas = (clave, fecha) con fecha ISO AAAA-MM-DD | 5.4 |
| Productos con descripciones distintas entre años | Descripción de referencia: la de la hoja 2026 | 5.4 |
| Solo 363 de los 691 vigentes aparecen en las cinco hojas | Se guardan los 691 con `historial_completo`; solo los 363 con valor 1 se analizan y se modelan (los 328 restantes son 91 nuevos y 237 con historia parcial) | 5.4 y 5.5 |
| Los ceros son ventas reales (19.8%) | No se imputan; se conservan | 5.4 |
| Octubre de 2026 incompleto | Corte en septiembre de 2026 (57 meses); octubre a diciembre son el horizonte | 5.4 y 5.5 |
| Abril y mayo de 2026 ajustados por el personal | Datos definitivos; en cargas futuras prevalece la última exportación | 5.4 |
| 11 productos en kilogramos | `unidad` = KG; se excluyen de los totales en piezas | 5.4 |
| 113 atípicos (96 aislados, 9 sostenidos, 8 recurrentes) | Se marcan sin borrarlos; propuesta: columna ajustada con la mediana solo para los aislados y el pedido del kit | 5.4 |
| Pedido del kit de diciembre de 2022 (confirmado) | Se marca como atípico: no representa demanda recurrente | 5.4 |
| Sostenidos y recurrentes | No se ajustan; se revisan como posibles cambios de nivel o estacionalidad | 5.4 y 5.5 |
| Ventas sin crecimiento sostenido en estos productos (−15.4% de enero a septiembre de 2026) | Los modelos parten de una demanda estable o a la baja, no de una tendencia creciente | 5.5 |
| Temporada alta y baja respaldada en parte | Propuesta: usar el mes como variable de calendario y revisar con la empresa enero, febrero y mayo antes de fijar la bandera | 5.4 |
| Semana Santa sin patrón mensual claro | Evaluarla como variable de calendario en el modelado | 5.4 y 5.5 |
| Demanda concentrada y con muchos ceros | Medir con error ponderado global y con MAPE solo en productos con ventas suficientes | 5.5 y 5.6 |